# ccdp — train the VMMRdb identifier on Colab (GDrive-backed, crash-resumable)

Same training loop as the Kaggle notebook (`ccdp train identifier-continue`), but built
for Google Colab. Checkpoints write straight to Google Drive every epoch, so a runtime
disconnect at epoch 9/12 doesn't lose work — the next launch auto-resumes from `last.pt`.

### Before you run
1. **Runtime → Change runtime type → GPU** (T4 free tier or A100 if you have Pro).
2. You'll be prompted to mount your Drive in section 2. Approve it.
3. You'll need a `kaggle.json` API token to pull the VMMRdb dataset
   (Kaggle → Account → Create New Token). Either:
   - drop it into `MyDrive/kaggle/kaggle.json` once, or
   - upload it via the file picker each session.

### Why GDrive?
Colab's `/content` is ephemeral — it dies with the VM. `/content/drive/MyDrive/...` is durable.
We point the run dir at GDrive so every `epoch_NNN.pt` lands on Drive immediately.

### Time + budget
~12 epochs on a T4 ≈ **6–10 h**. Free Colab sessions cap at ~12 h with idle disconnects;
Colab Pro avoids most of that. Either way, the resume logic below makes a mid-run kill cheap.


## 1. GPU check

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/ccdp'
CKPT_ROOT  = f'{DRIVE_ROOT}/checkpoints/identifier'   # durable
os.makedirs(CKPT_ROOT, exist_ok=True)
print('drive ckpt root:', CKPT_ROOT)

## 3. Clone the repo + install

We work out of `/content` (fast local SSD) but symlink `checkpoints/identifier` → Drive.

In [ ]:
%cd /content
!rm -rf car-crash-fix-amount-predictor
!git clone --depth 1 https://github.com/theDocWho/car-crash-fix-amount-predictor.git
%cd /content/car-crash-fix-amount-predictor
!pip -q install -e '.[ml]'

# Redirect the registry's identifier run dir at Drive so every saved epoch lands there.
import os, pathlib
pathlib.Path('checkpoints').mkdir(exist_ok=True)
local_link = pathlib.Path('checkpoints/identifier')
if local_link.exists() or local_link.is_symlink():
    local_link.unlink()
local_link.symlink_to(CKPT_ROOT)
print('symlinked checkpoints/identifier ->', os.readlink(local_link))

## 4. Fetch VMMRdb via Kaggle API

The CC0 mirror is `prabashwara/vmmrdb-dataset` (~291k images, ~9,170 classes). One-time:
upload your `kaggle.json` to Drive (`MyDrive/kaggle/kaggle.json`).

In [ ]:
import os, shutil, pathlib
KAGGLE_JSON_DRIVE = '/content/drive/MyDrive/kaggle/kaggle.json'
kdir = pathlib.Path.home() / '.kaggle'
kdir.mkdir(exist_ok=True)
if not (kdir / 'kaggle.json').exists():
    assert os.path.exists(KAGGLE_JSON_DRIVE), (
        f'put your kaggle API token at {KAGGLE_JSON_DRIVE} (or upload via Files panel)')
    shutil.copy(KAGGLE_JSON_DRIVE, kdir / 'kaggle.json')
    os.chmod(kdir / 'kaggle.json', 0o600)
!pip -q install kaggle

# Cache the dataset on Drive too — re-downloading 70GB every session is brutal.
DATA_ROOT_DRIVE = f'{DRIVE_ROOT}/data/raw/vmmrdb-dataset/prabashwara/vmmrdb-dataset'
os.makedirs(DATA_ROOT_DRIVE, exist_ok=True)
# Mirror to the loader's expected path:
os.makedirs('data/raw/vmmrdb-dataset/prabashwara', exist_ok=True)
local_data = pathlib.Path('data/raw/vmmrdb-dataset/prabashwara/vmmrdb-dataset')
if local_data.exists() or local_data.is_symlink():
    local_data.unlink()
local_data.symlink_to(DATA_ROOT_DRIVE)

# Only download if Drive copy is empty.
import glob
if not glob.glob(f'{DATA_ROOT_DRIVE}/*'):
    !kaggle datasets download -d prabashwara/vmmrdb-dataset -p {DATA_ROOT_DRIVE} --unzip
else:
    print('VMMRdb already cached on Drive — skipping download')

from ccdp.data import vmmrdb
counts = vmmrdb._class_dir_counts('data/raw/vmmrdb-dataset')
print(f'class folders: {len(counts)}  total images: {sum(counts.values())}')

## 5. Fetch the base 196-class identifier (warm-start)

In [ ]:
import os
!ccdp costing init || true
!mkdir -p checkpoints/production
if not os.path.exists('checkpoints/production/identifier.pt'):
    !curl -fL --retry 3 -o checkpoints/production/identifier.pt \
       https://github.com/theDocWho/car-crash-fix-amount-predictor/releases/download/v0.2.0/identifier.pt
print('base identifier:',
      os.path.exists('checkpoints/production/identifier.pt'),
      os.path.getsize('checkpoints/production/identifier.pt') if os.path.exists('checkpoints/production/identifier.pt') else 0,
      'bytes')

## 6. Continue-train with auto-resume

Scans the Drive-backed `checkpoints/identifier/` for prior `run_*identifier_vmmrdb/last.pt`.
If found, passes `--resume` and `--resume-run-dir` so the trainer picks up at epoch N+1
into the same run dir. Otherwise starts fresh.

In [ ]:
import glob, pathlib, shlex, os

prior = sorted(glob.glob('checkpoints/identifier/run_*identifier_vmmrdb'))
resume_flags = ''
if prior:
    run_dir = prior[-1]
    last_pt = pathlib.Path(run_dir) / 'last.pt'
    if last_pt.exists():
        resume_flags = f'--resume {shlex.quote(str(last_pt))} --resume-run-dir {shlex.quote(run_dir)}'
        print(f'[resume] picking up from {last_pt}')
    else:
        print(f'[resume] {run_dir} has no last.pt — starting fresh')
else:
    print('[resume] no prior runs — fresh start')

cmd = (
    'ccdp train identifier-continue '
    '--dataset vmmrdb --top-n 0 --batch-size 64 --num-workers 4 '
    f'{resume_flags}'
)
print(f'$ {cmd}')
os.system(cmd)

## 7. Promote + export

In [ ]:
import glob, shutil, os
src = sorted(glob.glob('checkpoints/identifier/run_*identifier_vmmrdb/best.pt'))
if src:
    real = os.path.realpath(src[-1])
    out = f'{DRIVE_ROOT}/identifier.pt'
    shutil.copy(real, out)
    print(f'exported -> {out}')
    !ccdp registry list --variant identifier | tail -5
else:
    print('no best.pt yet — check the training output above')

## 8. If the runtime crashed — resume flow

1. Reconnect a fresh runtime.
2. Re-run cells 1–6 top-to-bottom. Nothing to change.
3. Cell 2 remounts Drive. Cell 3 re-symlinks `checkpoints/identifier` → Drive.
4. Cell 6 detects the existing `run_*identifier_vmmrdb/last.pt` on Drive and adds
   `--resume <path> --resume-run-dir <dir>`.
5. Training loads model + epoch + stage + best_val from the checkpoint, re-applies the
   stage-2 unfreeze if needed, and writes new epochs into the same run dir.

A 9/12 crash → epoch 10 starts on next launch, no recompute. Cell 7's `best.pt` keeps
accumulating across sessions because `best.pt` is a symlink inside the same `run_*` dir
and the trainer only overwrites it when val-acc improves.

### Final step — push to GitHub release
On your local machine:
```bash
cp /content/drive/MyDrive/ccdp/identifier.pt .
gh release upload v0.2.0 identifier.pt --clobber
```
